# 00. 레시피 데이터 전처리 (Colab 버전)

## 목적
대용량 CSV(208K 레시피) 분할 처리 및 재료 정규화 파이프라인 구축

## 데이터 소스
- `TB_RECIPE_SEARCH_241226.csv`: 23,192개 (UTF-8)
- `TB_RECIPE_SEARCH-231130.csv`: 184,990개 (CP949)

## 출력
- `data/processed/recipe_v3/recipes_normalized.pkl`
- `data/processed/recipe_v3/ingredient_vocab.json`

## Colab 사용법
1. Google Drive 마운트
2. 프로젝트 폴더를 Drive에 업로드
3. 아래 셀 실행

In [1]:
# [1단계] Colab 환경 설정 및 Drive 마운트
import sys
import os

# Colab 환경 감지
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # 프로젝트 루트 설정 (본인의 Drive 경로에 맞게 수정)
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone'
    os.chdir(PROJECT_ROOT)

    # sys.path에 notebooks 경로 추가 (utils 모듈 import를 위해 필수!)
    sys.path.insert(0, f'{PROJECT_ROOT}/notebooks')

    print(f"✅ Colab 환경 감지")
    print(f"📁 프로젝트 루트: {PROJECT_ROOT}")
    print(f"📁 sys.path 추가됨: {PROJECT_ROOT}/notebooks")
else:
    from pathlib import Path
    PROJECT_ROOT = Path().resolve().parent
    sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))

    print("💻 로컬 환경에서 실행 중")
    print(f"📁 프로젝트 루트: {PROJECT_ROOT}")

Mounted at /content/drive
✅ Colab 환경 감지
📁 프로젝트 루트: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone
📁 sys.path 추가됨: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/notebooks


In [2]:
# [1.5단계] 경로 확인 (디버그용 - 문제 발생 시 실행)
# utils 폴더가 제대로 있는지 확인
import os

print("=== 경로 디버그 ===")
print(f"현재 작업 디렉토리: {os.getcwd()}")
print(f"sys.path: {sys.path[:3]}...")

# notebooks/utils 폴더 확인
notebooks_path = f"{PROJECT_ROOT}/notebooks" if IN_COLAB else str(PROJECT_ROOT / 'notebooks')
utils_path = f"{notebooks_path}/utils"

print(f"\nnotebooks 경로: {notebooks_path}")
print(f"notebooks 존재: {os.path.exists(notebooks_path)}")

if os.path.exists(notebooks_path):
    print(f"notebooks 내용: {os.listdir(notebooks_path)[:10]}")

print(f"\nutils 경로: {utils_path}")
print(f"utils 존재: {os.path.exists(utils_path)}")

if os.path.exists(utils_path):
    print(f"utils 내용: {os.listdir(utils_path)}")

=== 경로 디버그 ===
현재 작업 디렉토리: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone
sys.path: ['/content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/notebooks', '/content', '/env/python']...

notebooks 경로: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/notebooks
notebooks 존재: True
notebooks 내용: ['05_evaluation_export.ipynb', '00_recipe_preprocessing.ipynb', '01_EDA_recipe_analysis.ipynb', '02_feature_engineering.ipynb', '03_validation_strategy.ipynb', 'logs', '04_modeling_training.ipynb', 'utils']

utils 경로: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/notebooks/utils
utils 존재: True
utils 내용: ['__pycache__', 'gap_filling', 'data_loader.py', 'recommender_models.py', 'weight_config.py', 'model_exporter.py', 'kaggle_optimizations.py', 'optimized_als_recommender.py', '__init__.py']


In [3]:
# [2단계] 라이브러리 임포트
import gc
import json
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from tqdm.auto import tqdm  # Colab 호환

# Colab에서는 문자열을 Path로 변환
if IN_COLAB:
    PROJECT_ROOT = Path(PROJECT_ROOT)

# Gap Filling 유틸리티 임포트
from utils.gap_filling import IngredientParser, MultiStageNormalizer

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"Colab 환경: {IN_COLAB}")

프로젝트 루트: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone
Colab 환경: True


In [4]:
# 경로 설정
DATA_DIR = PROJECT_ROOT / 'data' / 'recipe'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'recipe_v3'

# 출력 디렉토리 생성
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 데이터 파일 경로
CSV_2024 = DATA_DIR / 'TB_RECIPE_SEARCH_241226.csv'  # UTF-8
CSV_2023 = DATA_DIR / 'TB_RECIPE_SEARCH-231130.csv'  # CP949

print(f"데이터 디렉토리: {DATA_DIR}")
print(f"출력 디렉토리: {OUTPUT_DIR}")
print(f"\n2024 데이터 존재: {CSV_2024.exists()}")
print(f"2023 데이터 존재: {CSV_2023.exists()}")

데이터 디렉토리: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/recipe
출력 디렉토리: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3

2024 데이터 존재: True
2023 데이터 존재: True


## 1. 데이터 로드 (Chunk 단위)

In [5]:
def load_csv_chunked(
    file_path: Path,
    encoding: str = 'utf-8',
    chunksize: int = 10000
) -> pd.DataFrame:
    """대용량 CSV를 청크 단위로 로드

    Args:
        file_path: CSV 파일 경로
        encoding: 파일 인코딩
        chunksize: 청크 크기

    Returns:
        병합된 DataFrame
    """
    chunks = []

    for chunk in tqdm(
        pd.read_csv(
            file_path,
            encoding=encoding,
            encoding_errors='replace',  # 디코딩 불가 문자는 � 로 대체
            chunksize=chunksize,
            on_bad_lines='skip'
        ),
        desc=f"로딩 {file_path.name}"
    ):
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    return df

In [6]:
# 2024 데이터 로드 (UTF-8)
if CSV_2024.exists():
    df_2024 = load_csv_chunked(CSV_2024, encoding='utf-8')
    print(f"2024 데이터: {len(df_2024):,}개 레시피")
    print(f"컬럼: {df_2024.columns.tolist()}")
else:
    df_2024 = pd.DataFrame()
    print("2024 데이터 파일 없음")

로딩 TB_RECIPE_SEARCH_241226.csv: 0it [00:00, ?it/s]

2024 데이터: 23,192개 레시피
컬럼: ['RCP_SNO', 'RCP_TTL', 'CKG_NM', 'RGTR_ID', 'RGTR_NM', 'INQ_CNT', 'RCMM_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM', 'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM', 'CKG_IPDC', 'CKG_MTRL_CN', 'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM', 'FIRST_REG_DT', 'RCP_IMG_URL']


In [7]:
# 2023 데이터 로드 (CP949)
if CSV_2023.exists():
    df_2023 = load_csv_chunked(CSV_2023, encoding='cp949')
    print(f"2023 데이터: {len(df_2023):,}개 레시피")
    print(f"컬럼: {df_2023.columns.tolist()}")
else:
    df_2023 = pd.DataFrame()
    print("2023 데이터 파일 없음")

로딩 TB_RECIPE_SEARCH-231130.csv: 0it [00:00, ?it/s]

2023 데이터: 184,991개 레시피
컬럼: ['RCP_SNO', 'RCP_TTL', 'CKG_NM', 'RGTR_ID', 'RGTR_NM', 'INQ_CNT', 'RCMM_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM', 'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM', 'CKG_IPDC', 'CKG_MTRL_CN', 'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM', 'FIRST_REG_DT']


In [8]:
# 데이터 병합 및 중복 제거
print("\n=== 데이터 병합 ===")

# 공통 컬럼 확인
if len(df_2024) > 0 and len(df_2023) > 0:
    common_cols = list(set(df_2024.columns) & set(df_2023.columns))
    print(f"공통 컬럼: {common_cols}")

    # 공통 컬럼만 사용하여 병합
    df_combined = pd.concat([
        df_2024[common_cols],
        df_2023[common_cols]
    ], ignore_index=True)

    # 메모리 해제
    del df_2024, df_2023
    gc.collect()

elif len(df_2024) > 0:
    df_combined = df_2024
    del df_2024
else:
    df_combined = df_2023
    del df_2023

print(f"병합 전 레시피 수: {len(df_combined):,}")

# 레시피 ID 기준 중복 제거
if 'RCP_ID' in df_combined.columns:
    df_combined = df_combined.drop_duplicates(subset=['RCP_ID'])
    print(f"중복 제거 후 레시피 수: {len(df_combined):,}")

gc.collect()


=== 데이터 병합 ===
공통 컬럼: ['RCP_SNO', 'CKG_MTH_ACTO_NM', 'CKG_STA_ACTO_NM', 'CKG_INBUN_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM', 'INQ_CNT', 'RCP_TTL', 'CKG_MTRL_CN', 'CKG_IPDC', 'CKG_NM', 'CKG_DODF_NM', 'FIRST_REG_DT', 'RCMM_CNT', 'RGTR_ID', 'CKG_TIME_NM', 'SRAP_CNT', 'RGTR_NM']
병합 전 레시피 수: 208,183


0

In [9]:
# 데이터 샘플 확인
print("\n=== 데이터 샘플 ===")
df_combined.head(3)


=== 데이터 샘플 ===


,RCP_SNO,CKG_MTH_ACTO_NM,CKG_STA_ACTO_NM,CKG_INBUN_NM,CKG_MTRL_ACTO_NM,CKG_KND_ACTO_NM,INQ_CNT,RCP_TTL,CKG_MTRL_CN,CKG_IPDC,CKG_NM,CKG_DODF_NM,FIRST_REG_DT,RCMM_CNT,RGTR_ID,CKG_TIME_NM,SRAP_CNT,RGTR_NM
0,7016813,끓이기,명절,2인분,소고기,국/탕,743,멸치육수 소고기 떡국 만드는법,[재료] 떡국떡400g| 다진소고기100g| 멸치육수800ml| 대...,새해가 되면 뜨끈한 떡국 한 그릇이 생각나는데요. 오늘은 집에서 간단하게 요리할 수...,소고기떡국,초급,20240101000857,0,ranch6356,NaN,2,반이짝이
1,7016814,삶기,술안주,2인분,돼지고기,메인반찬,1396,#수육용삼겹살 #된장수육만들기 #일상먹거리 #무생채와함께먹는된장수육,[재료] 돼지고기 수육용삼겹살500g| 된장1.5큰술| 술4큰술| ...,수육용 삼겹살을 사다가 된장과 술을 넣고 일상먹거리 수육한접시를 만들어 주었습니다....,된장수육,아무나,20240101002917,0,kstencil,2시간이내,1,강철새잎
2,7016815,끓이기,해장,4인분,돼지고기,국/탕,4008,우거지감자탕 뼈해장국 끓이는법,[재료] 돼지등뼈1.5kg| 양파1/2개| 감자1개| 대파1대...,까다로운 남편의 입맛을 맞추기위해 여러번 시도끝에 만들어낸 최적의 레시피입니다. 한...,우거지감자탕,중급,20240101020501,0,87771622,2시간이내,29,김한솔


## 2. 재료 파싱 및 정규화

In [10]:
# 재료 컬럼 확인
INGREDIENT_COL = 'CKG_MTRL_CN'  # 재료 정보 컬럼

if INGREDIENT_COL in df_combined.columns:
    print(f"재료 컬럼 '{INGREDIENT_COL}' 존재")
    print(f"\n샘플 재료 데이터:")
    for idx, val in enumerate(df_combined[INGREDIENT_COL].dropna().head(3)):
        print(f"\n[{idx+1}] {val[:200]}...")
else:
    print(f"재료 컬럼 '{INGREDIENT_COL}' 없음")
    print(f"사용 가능한 컬럼: {df_combined.columns.tolist()}")

재료 컬럼 'CKG_MTRL_CN' 존재

샘플 재료 데이터:

[1] [재료] 떡국떡400g| 다진소고기100g| 멸치육수800ml| 대파1/3대| 계란2개| 참기름1T| 국간장1T| 참기름1/2T| 다진마늘1t| 소금| 김가루약간...

[2] [재료] 돼지고기 수육용삼겹살500g| 된장1.5큰술| 술4큰술| 홍어무침| 무생채| 콩나물무침...

[3] [재료] 돼지등뼈1.5kg| 양파1/2개| 감자1개| 대파1대| 알배기배추1/2개| 청양고추2개| 깻잎10~15장 [양념] 된장2T| 고추장2T| 다진마늘1T| 간장3T| 고춧가루3T| 액젓3T| 다진생강1t| 다시다1t| 들깨가루3T...


In [11]:
# 파서 및 정규화기 초기화
parser = IngredientParser()
normalizer = MultiStageNormalizer()

print("파서 및 정규화기 초기화 완료")

파서 및 정규화기 초기화 완료


In [12]:
# 파싱 테스트
test_sample = df_combined[INGREDIENT_COL].dropna().iloc[0]
print(f"원본: {test_sample[:100]}...\n")

parsed = parser.parse(test_sample)
print(f"파싱 결과 ({len(parsed)}개 재료):")
for item in parsed[:5]:
    print(f"  - {item}")

원본: [재료] 떡국떡400g| 다진소고기100g| 멸치육수800ml| 대파1/3대| 계란2개| 참기름1T| 국간장1T| 참기름1/2T| 다진마...

파싱 결과 (11개 재료):
  - {'name': '떡국떡\x07\x07g\x07', 'quantity': '400', 'category': '재료'}
  - {'name': '다진소고기\x07\x07g\x07', 'quantity': '100', 'category': '재료'}
  - {'name': '멸치육수\x07\x07ml\x07', 'quantity': '800', 'category': '재료'}
  - {'name': '대파\x07\x07대\x07', 'quantity': '1/3', 'category': '재료'}
  - {'name': '계란\x07\x07개\x07', 'quantity': '2', 'category': '재료'}


In [13]:
# 정규화 테스트
ingredient_names = [item['name'] for item in parsed]
print(f"\n원본 재료: {ingredient_names[:5]}")

normalized = normalizer.normalize_list(ingredient_names)
print(f"정규화 후: {normalized[:5]}")


원본 재료: ['떡국떡\x07\x07g\x07', '다진소고기\x07\x07g\x07', '멸치육수\x07\x07ml\x07', '대파\x07\x07대\x07', '계란\x07\x07개\x07']
정규화 후: ['떡국떡\x07\x07g\x07', '소고기\x07\x07g\x07', '멸치육수\x07\x07ml\x07', '파', '계란\x07\x07개\x07']


In [14]:
def process_recipe(row, parser, normalizer, ingredient_col='RCP_PARTS_DTLS'):
    """단일 레시피 처리

    Returns:
        정규화된 재료 리스트 또는 None
    """
    raw = row.get(ingredient_col, '')

    if pd.isna(raw) or not raw.strip():
        return None

    try:
        # 파싱
        parsed = parser.parse(raw)
        if not parsed:
            return None

        # 재료명 추출
        ingredient_names = [item['name'] for item in parsed]

        # 정규화
        normalized = normalizer.normalize_list(ingredient_names)

        # 빈 값 및 중복 제거
        normalized = list(dict.fromkeys([n for n in normalized if n]))

        return normalized if len(normalized) >= 2 else None

    except Exception as e:
        return None

In [15]:
# 전체 레시피 처리
print("=== 전체 레시피 처리 ===")

processed_recipes = []
recipe_ids = []
recipe_names = []

for idx, row in tqdm(df_combined.iterrows(), total=len(df_combined), desc="레시피 처리"):
    ingredients = process_recipe(row.to_dict(), parser, normalizer, ingredient_col=INGREDIENT_COL)

    if ingredients:
        processed_recipes.append(ingredients)
        recipe_ids.append(row.get('RCP_SNO', idx))  # RCP_ID → RCP_SNO
        recipe_names.append(row.get('RCP_TTL', f'recipe_{idx}'))  # RCP_NM → RCP_TTL

print(f"\n유효 레시피 수: {len(processed_recipes):,} / {len(df_combined):,}")
print(f"성공률: {len(processed_recipes)/len(df_combined)*100:.1f}%")

=== 전체 레시피 처리 ===


레시피 처리:   0%|          | 0/208183 [00:00<?, ?it/s]


유효 레시피 수: 205,012 / 208,183
성공률: 98.5%


In [16]:
# 처리 결과 확인
print("\n=== 처리 결과 샘플 ===")
for i in range(min(5, len(processed_recipes))):
    print(f"\n[{recipe_names[i]}]")
    print(f"  재료: {processed_recipes[i]}")


=== 처리 결과 샘플 ===

[멸치육수 소고기 떡국 만드는법]
  재료: ['떡국떡\x07\x07g\x07', '소고기\x07\x07g\x07', '멸치육수\x07\x07ml\x07', '파', '계란\x07\x07개\x07', '참기름\x07\x07T\x07', '간장', '마늘\x07\x07t\x07', '소금\x07\x07\x07', '김가루\x07\x07약간\x07']

[#수육용삼겹살 #된장수육만들기 #일상먹거리 #무생채와함께먹는된장수육]
  재료: ['돼지고기', '된장\x07\x07큰술\x07', '술\x07\x07큰술\x07', '홍어무침\x07\x07\x07', '무생채\x07\x07\x07', '콩나물무침\x07\x07\x07']

[우거지감자탕 뼈해장국 끓이는법]
  재료: ['돼지고기', '양파\x07\x07개\x07', '감자', '파', '배추', '고추', '깻잎\x07~\x07장\x07 된장\x07\x07T\x07', '고추장\x07\x07T\x07', '마늘\x07\x07T\x07', '장\x07\x07T\x07', '고추가루', '액젓\x07\x07T\x07', '생강\x07\x07t\x07', '다시다\x07\x07t\x07', '들깨가루\x07\x07T\x07']

[만두전골 레시피 백종원 만두 전골요리 뜨끈하고 진한 국물이 일품]
  재료: ['만두\x07\x07개\x07', '배추', '양파\x07\x07개\x07', '파', '버섯', '고추', '다시마\x07\x07개\x07', '고추가루', '간장', '생선', '마늘\x07\x07T\x07', '후추']

[새해 통삼겹살 무수분 보쌈 삶는법 백종원 보쌈 마늘소스 만들기]
  재료: ['돼지고기', '양파\x07\x07개\x07', '못난이 사과 小\x07\x07개\x07', '파', '마늘\x07\x07개\x07', '강가루\x07\x07\x07', '월계수 잎\x07~\x07장\x07', '후추', '맛술\x07\x07컵\x07 마늘\x07\x07T\x07

## 3. 재료 통계 분석

In [17]:
# 재료 빈도 계산
ingredient_counter = Counter()
for recipe in processed_recipes:
    ingredient_counter.update(recipe)

print(f"\n=== 재료 통계 ===")
print(f"총 고유 재료 수: {len(ingredient_counter):,}")

# 레시피당 재료 수 통계
recipe_lengths = [len(r) for r in processed_recipes]
print(f"\n레시피당 재료 수:")
print(f"  - 평균: {np.mean(recipe_lengths):.1f}")
print(f"  - 중앙값: {np.median(recipe_lengths):.1f}")
print(f"  - 최소: {min(recipe_lengths)}")
print(f"  - 최대: {max(recipe_lengths)}")

# 상위 재료
print(f"\n상위 20개 재료:")
for ing, count in ingredient_counter.most_common(20):
    print(f"  {ing}: {count:,}회 ({count/len(processed_recipes)*100:.1f}%)")


=== 재료 통계 ===
총 고유 재료 수: 127,854

레시피당 재료 수:
  - 평균: 8.4
  - 중앙값: 8.0
  - 최소: 2
  - 최대: 57

상위 20개 재료:
  파: 66,607회 (32.5%)
  마늘: 61,398회 (29.9%)
  양파: 51,700회 (25.2%)
  간장: 51,496회 (25.1%)
  설탕: 47,440회 (23.1%)
  고추가루: 45,570회 (22.2%)
  고추: 45,303회 (22.1%)
  소금: 45,083회 (22.0%)
  계란: 39,149회 (19.1%)
  후추: 38,223회 (18.6%)
  참기름: 35,588회 (17.4%)
  생크림: 29,342회 (14.3%)
  버섯: 22,599회 (11.0%)
  식용유: 20,785회 (10.1%)
  당근: 20,474회 (10.0%)
  통깨: 20,190회 (9.8%)
  생선: 18,761회 (9.2%)
  조개: 18,015회 (8.8%)
  돼지고기: 16,684회 (8.1%)
  감자: 15,789회 (7.7%)


In [18]:
# 빈도 분포 (Head/Torso/Tail)
freq_list = list(ingredient_counter.values())

head_threshold = 1000  # 1000회 이상
torso_threshold = 100  # 100회 이상

head_count = sum(1 for f in freq_list if f >= head_threshold)
torso_count = sum(1 for f in freq_list if torso_threshold <= f < head_threshold)
tail_count = sum(1 for f in freq_list if f < torso_threshold)

print(f"\n=== 빈도 분포 (Zipf's Law) ===")
print(f"Head (>={head_threshold}회): {head_count}개 ({head_count/len(ingredient_counter)*100:.1f}%)")
print(f"Torso ({torso_threshold}-{head_threshold-1}회): {torso_count}개 ({torso_count/len(ingredient_counter)*100:.1f}%)")
print(f"Tail (<{torso_threshold}회): {tail_count}개 ({tail_count/len(ingredient_counter)*100:.1f}%)")


=== 빈도 분포 (Zipf's Law) ===
Head (>=1000회): 139개 (0.1%)
Torso (100-999회): 896개 (0.7%)
Tail (<100회): 126819개 (99.2%)


## 4. 희귀 재료 필터링

In [19]:
# 최소 빈도 필터링
MIN_FREQ = 5  # 최소 5회 이상 등장

valid_ingredients = set(
    ing for ing, count in ingredient_counter.items()
    if count >= MIN_FREQ
)

print(f"\n=== 희귀 재료 필터링 ===")
print(f"필터링 전 재료 수: {len(ingredient_counter):,}")
print(f"필터링 후 재료 수: {len(valid_ingredients):,}")
print(f"제거된 희귀 재료: {len(ingredient_counter) - len(valid_ingredients):,}")


=== 희귀 재료 필터링 ===
필터링 전 재료 수: 127,854
필터링 후 재료 수: 12,132
제거된 희귀 재료: 115,722


In [20]:
# 필터링 적용
filtered_recipes = []
filtered_ids = []
filtered_names = []

for recipe, rcp_id, rcp_name in zip(processed_recipes, recipe_ids, recipe_names):
    # 유효한 재료만 유지
    filtered = [ing for ing in recipe if ing in valid_ingredients]

    # 최소 2개 이상의 재료가 있어야 유효
    if len(filtered) >= 2:
        filtered_recipes.append(filtered)
        filtered_ids.append(rcp_id)
        filtered_names.append(rcp_name)

print(f"\n필터링 후 레시피 수: {len(filtered_recipes):,}")
print(f"제거된 레시피: {len(processed_recipes) - len(filtered_recipes):,}")


필터링 후 레시피 수: 202,181
제거된 레시피: 2,831


## 5. 데이터 저장

In [21]:
# DataFrame 생성
df_output = pd.DataFrame({
    'recipe_id': filtered_ids,
    'recipe_name': filtered_names,
    'ingredients': filtered_recipes,
    'num_ingredients': [len(r) for r in filtered_recipes]
})

print(f"출력 DataFrame 형태: {df_output.shape}")
df_output.head()

출력 DataFrame 형태: (202181, 4)


,recipe_id,recipe_name,ingredients,num_ingredients
0,7016813,멸치육수 소고기 떡국 만드는법,"[떡국떡g, 소고기g, 멸치육수ml, 파, 계란개, 참기름...",10
1,7016814,#수육용삼겹살 #된장수육만들기 #일상먹거리 #무생채와함께먹는된장수육,"[돼지고기, 된장큰술]",2
2,7016815,우거지감자탕 뼈해장국 끓이는법,"[돼지고기, 양파개, 감자, 파, 배추, 고추, 고추장T, 마늘T,...",14
3,7016816,만두전골 레시피 백종원 만두 전골요리 뜨끈하고 진한 국물이 일품,"[만두개, 배추, 양파개, 파, 버섯, 고추, 다시마개, 고추가루,...",12
4,7016817,새해 통삼겹살 무수분 보쌈 삶는법 백종원 보쌈 마늘소스 만들기,"[돼지고기, 양파개, 파, 마늘개, 강가루, 후추, 설탕T, ...",10


In [22]:
# Pickle로 저장 (PyArrow 버전 충돌 방지)
import pickle

output_path = OUTPUT_DIR / 'recipes_normalized.pkl'
df_output.to_pickle(output_path)
print(f"레시피 저장 완료: {output_path}")
print(f"파일 크기: {output_path.stat().st_size / 1024 / 1024:.1f} MB")

레시피 저장 완료: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/recipes_normalized.pkl
파일 크기: 29.0 MB


In [23]:
# 재료 어휘 저장
vocab_data = {
    'ingredients': list(valid_ingredients),
    'frequencies': {ing: ingredient_counter[ing] for ing in valid_ingredients},
    'total_recipes': len(filtered_recipes),
    'min_freq': MIN_FREQ,
    'version': '3.0.0'
}

vocab_path = OUTPUT_DIR / 'ingredient_vocab.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)

print(f"어휘 저장 완료: {vocab_path}")

어휘 저장 완료: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/ingredient_vocab.json


In [24]:
# 최종 요약
print("\n" + "="*50)
print("전처리 완료 요약")
print("="*50)
print(f"원본 레시피 수: {len(df_combined):,}")
print(f"최종 레시피 수: {len(filtered_recipes):,}")
print(f"유효 재료 수: {len(valid_ingredients):,}")
print(f"\n출력 파일:")
print(f"  - {OUTPUT_DIR / 'recipes_normalized.pkl'}")
print(f"  - {vocab_path}")


전처리 완료 요약
원본 레시피 수: 208,183
최종 레시피 수: 202,181
유효 재료 수: 12,132

출력 파일:
  - /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/recipes_normalized.pkl
  - /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/ingredient_vocab.json


In [25]:
# 메모리 정리
del df_combined, processed_recipes, filtered_recipes
gc.collect()
print("메모리 정리 완료")

메모리 정리 완료
